In [3]:
!pip install pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 23.0 MB/s eta 0:00:00m eta 0:00:010:0101
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 23.9 MB/s eta 0:00:00m eta 0:00:010:00:01


In [4]:
import pandas as pd

In [23]:
df_posts = pd.read_csv('Social_Engine_Posts_Corrupted.csv')
df_users = pd.read_csv('Social_Engine_Users.csv')

In [24]:
def parse_timestamp(value):
    if pd.isna(value):
        return pd.NaT

    value = str(value).strip()

    # Unix timestamp
    if value.isdigit():
        return pd.to_datetime(int(value), unit='s')

    # Date-only format: DD-MM-YYYY
    if len(value) == 10 and value[2] == '-' and value[5] == '-':
        return pd.to_datetime(value, format='%d-%m-%Y', errors='coerce')

    # ISO / other datetime formats
    return pd.to_datetime(value, errors='coerce')


df_posts['timestamp_parsed'] = df_posts['timestamp'].apply(parse_timestamp)

# Final standardized formats
df_posts['date'] = df_posts['timestamp_parsed'].dt.strftime('%Y-%m-%d')
df_posts['time'] = df_posts['timestamp_parsed'].dt.strftime('%H:%M:%S')

# Original date-only timestamps have no time information
date_only = (
    df_posts['timestamp'].notna() &
    df_posts['timestamp'].astype(str).str.match(r'^\d{2}-\d{2}-\d{4}$')
)

df_posts.loc[date_only, 'time'] = pd.NA

# Remove temporary/original timestamp columns
df_posts.drop(
    columns=['timestamp', 'timestamp_parsed'],
    inplace=True
)

In [25]:
# Convert engagement counts to nullable integers
df_posts['likes'] = df_posts['likes'].astype('Int64')
df_posts['shares'] = df_posts['shares'].astype('Int64')
df_posts['comments'] = df_posts['comments'].astype('Int64')

# Check negative values
print("Negative likes:", (df_posts['likes'] < 0).sum())
print("Negative shares:", (df_posts['shares'] < 0).sum())
print("Negative comments:", (df_posts['comments'] < 0).sum())

# Check missing values
print("\nMissing values:")
print(df_posts.isna().sum())

# Check duplicate post IDs
print("\nDuplicate post IDs:", df_posts['post_id'].duplicated().sum())

Negative likes: 525
Negative shares: 0
Negative comments: 0

Missing values:
post_id            0
user_id            0
platform        1846
text_content    1746
likes           1858
shares             0
comments           0
date               0
time            3622
dtype: int64

Duplicate post IDs: 360


In [26]:
duplicates = df_posts[df_posts['post_id'].duplicated(keep=False)]

duplicates.sort_values('post_id').head(20)

,post_id,user_id,platform,text_content,likes,shares,comments,date,time
10546,01jzjra6mmwc,user_bqp8mrav,YouTube,Comparing Toyota Corolla to the competition. N...,2829,776,654,2024-10-13,15:19:29
3003,01jzjra6mmwc,user_bqp8mrav,YouTube,Comparing Toyota Corolla to the competition. N...,2829,776,654,2024-10-13,15:19:29
6645,03gqkhd2ebdg,user_ysbkmmxf,Facebook,My two days review of Adidas Stan Smith: Worth...,<NA>,966,833,2024-10-20,20:14:36
11348,03gqkhd2ebdg,user_ysbkmmxf,Facebook,My two days review of Adidas Stan Smith: Worth...,<NA>,966,833,2024-10-20,20:14:36
4324,03qci0ig2dz1,user_oki4xfhu,Instagram,Nike WinterWonders is amazing! Can't wait to s...,3506,379,449,2024-10-26,20:54:04
5671,03qci0ig2dz1,user_oki4xfhu,Instagram,Nike WinterWonders is amazing! Can't wait to s...,3506,379,449,2024-10-26,20:54:04
3983,06h8cm4e5ouf,user_hq1akhdd,YouTube,Has anyone else experienced customer service w...,<NA>,1204,470,2025-03-31,02:18:50
2633,06h8cm4e5ouf,user_hq1akhdd,YouTube,Has anyone else experienced customer service w...,<NA>,1204,470,2025-03-31,02:18:50
4032,06rvkdi7ipfz,user_2te8mebn,YouTube,Should I upgrade about Pepsi's Pepsi Zero Suga...,4950,325,953,2024-12-30,13:28:02
9153,06rvkdi7ipfz,user_2te8mebn,YouTube,Should I upgrade about Pepsi's Pepsi Zero Suga...,4950,325,953,2024-12-30,13:28:02


In [27]:
df_posts = df_posts.drop_duplicates(subset='post_id', keep='first')

In [30]:
# Convert dates to datetime
df_users['account_created'] = pd.to_datetime(
    df_users['account_created'],
    format='%Y-%m-%d',
    errors='coerce'
)

df_posts['date_check'] = pd.to_datetime(
    df_posts['date'],
    format='%Y-%m-%d',
    errors='coerce'
)

# Check for invalid user IDs
invalid_user = ~df_posts['user_id'].isin(df_users['user_id'])

print("Invalid user IDs:")
print(df_posts.loc[invalid_user, [
    'post_id',
    'user_id',
    'date',
    'platform',
    'text_content',
    'likes',
    'shares',
    'comments'
]].to_string(index=False))

print("\nNumber of invalid user IDs:", invalid_user.sum())

# Match posts with account creation dates
df_check = df_posts.merge(
    df_users[['user_id', 'account_created']],
    on='user_id',
    how='left'
)

# Check for posts before account creation
invalid_date = (
    df_check['account_created'].notna() &
    df_check['date_check'].notna() &
    (df_check['date_check'] < df_check['account_created'])
)

print("\nPosts before account creation:")
print(df_check.loc[invalid_date, [
    'post_id',
    'user_id',
    'date',
    'account_created',
    'platform',
    'text_content',
    'likes',
    'shares',
    'comments'
]].to_string(index=False))

print("\nNumber of posts before account creation:", invalid_date.sum())

# Combine invalid records
invalid_records = invalid_user | invalid_date

print("\nTotal records to delete:", invalid_records.sum())

# Delete invalid records
df_posts = df_posts.loc[~invalid_records].copy()

# Remove temporary column
df_posts.drop(columns=['date_check'], inplace=True)


Invalid user IDs:
Empty DataFrame
Columns: [post_id, user_id, date, platform, text_content, likes, shares, comments]
Index: []

Number of invalid user IDs: 0

Posts before account creation:
Empty DataFrame
Columns: [post_id, user_id, date, account_created, platform, text_content, likes, shares, comments]
Index: []

Number of posts before account creation: 0

Total records to delete: 0


In [35]:
print(df_posts['platform'].value_counts(dropna=False))
print(df_posts['text_content'].head(20).to_string())

platform
Facebook     2074
YouTube      2073
Twitter      2049
Reddit       2031
Instagram    1989
NaN          1784
Name: count, dtype: int64
0     Bummed out with my new Air Max from Nike! Abso...
1     My one month review of Pepsi Crystal Pepsi: Hi...
2     Just unboxed my new Highlander from Toyota. Ex...
3     Comparing Pepsi Crystal Pepsi to the competiti...
4     My one week review of Coca-Cola Diet Coke: Bes...
5     Adidas ValentinesDeals is disappointing! Can't...
6     Fed up with my new iPhone 15 from Apple! Best ...
7     Just unboxed my new React from Nike. Not bad. ...
8     Just saw an ad for Toyota Highlander during th...
9     Just unboxed my new iMac from Apple. Disappoin...
10    Just saw an ad for Samsung Galaxy Buds during ...
11    Amazon LoyaltyRewards is amazing! Can't wait t...
12    Has anyone else experienced customer service w...
13    Comparing Coca-Cola Fanta to the competition. ...
14    Just unboxed my new Halo Band from Amazon. Exc...
15               

In [36]:
# Convert negative likes to positive
df_posts['likes'] = df_posts['likes'].abs()

In [37]:
import re
import html

In [38]:
# Clean HTML and whitespace from text
def clean_text(value):
    if pd.isna(value):
        return value

    value = str(value)

    # Decode HTML entities such as &amp;
    value = html.unescape(value)

    # Remove HTML tags such as <br>, <div>, etc.
    value = re.sub(r'<[^>]+>', ' ', value)

    # Remove extra whitespace
    value = re.sub(r'\s+', ' ', value).strip()

    return value

df_posts['text_content'] = df_posts['text_content'].apply(clean_text)

In [43]:
# Remove trailing ampersand
df_posts['text_content'] = df_posts['text_content'].str.rstrip('&').str.rstrip()

In [44]:
df_posts

,post_id,user_id,platform,text_content,likes,shares,comments,date,time
0,to64mgey2v3y,user_vfxs1pry,Reddit,Bummed out with my new Air Max from Nike! Abso...,4488,1456,673,2024-09-25,NaN
1,7f0wdauzbj89,user_8l7rv5oe,Reddit,My one month review of Pepsi Crystal Pepsi: Hi...,789,1484,39,2024-08-01,16:14:00
2,dvvhg8eel45x,user_nfo3ih5u,NaN,Just unboxed my new Highlander from Toyota. Ex...,<NA>,1410,839,2025-04-13,20:12:18
3,hgb9cxke4t7b,user_27aje6ur,Facebook,Comparing Pepsi Crystal Pepsi to the competiti...,167,584,833,2024-09-10,NaN
4,zwerpw3wk320,user_9px5q0by,Reddit,My one week review of Coca-Cola Diet Coke: Bes...,4749,152,630,2024-05-31,NaN
...,...,...,...,...,...,...,...,...,...
12355,7nar7yotf6u4,user_zeija5lf,Facebook,NaN,<NA>,1663,870,2024-11-25,12:57:51
12356,qokoc4of26wk,user_nd1s5fa0,Facebook,Just unboxed my new Highlander from Toyota. Di...,<NA>,1258,765,2025-02-23,NaN
12357,jraym4uws51h,user_w66b5uk0,YouTube,NaN,2515,1355,36,2025-02-19,00:17:20
12358,21idyx41b5v4,user_tmjtxubu,Instagram,Just unboxed my new Vision Pro from Apple. Wor...,2920,978,235,2025-03-18,14:58:38


In [46]:
df_posts.to_csv(
    'Social_Engine_Posts_Cleaned.csv',
    index=False,
    na_rep='NULL'
)